In [17]:
import os
import json
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

In [21]:
CLASS_MAP = {
    "background": 0,           # 배경
    "crack": 1,                # 단순 균열
    "leak": 2,                 # 누수
    "efflorescence": 3,        # 백태
    "detachment": 4,           # 들뜸
    "reticular crack": 5,      # 망상 균열
    "spalling": 6,             # 박리
    "material separation": 7,  # 재료 분리
    "rebar": 8,                # 철근 노출
    "damage": 9,               # 파손
    "exhilaration": 10         # 박락
}

def generate_validation_masks(json_base_dir, output_mask_dir):
    os.makedirs(output_mask_dir, exist_ok=True)
    
    json_paths = list(Path(json_base_dir).rglob("*.json"))
    print(f"{len(json_paths):,}files")
    
    mask_count = 0
    
    for json_path in tqdm(json_paths):
        with open(json_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                continue
        
        img_info = data.get("image", {})
        if img_info.get("object_included", "N") == "N":
            continue
            
        annotations = img_info.get("annotations", [])
        if not annotations:
            continue
            
        width = img_info.get("width", 1920)
        height = img_info.get("height", 1080)
        mask = np.zeros((height, width), dtype=np.uint8)
        
        for ann in annotations:
            label_name = ann.get("label", "").lower()
            class_id = CLASS_MAP.get(label_name, 0)
            
            if class_id == 0: continue
                
            points = ann.get("points", [])
            if not points: continue
            
            pts = np.array(points, np.int32).reshape((-1, 1, 2))
            shape_type = ann.get("shape", "Polygon")
            
            if shape_type == "Polyline":
                cv2.polylines(mask, [pts], isClosed=False, color=class_id, thickness=5)
            else:
                cv2.fillPoly(mask, [pts], color=class_id)
                
        original_img_name = img_info.get("name", json_path.stem + ".jpg")
        mask_filename = Path(original_img_name).stem + ".png"
        mask_save_path = os.path.join(output_mask_dir, mask_filename)
        
        cv2.imwrite(mask_save_path, mask)
        mask_count += 1

    print(f"{mask_count:,}장")
    print(f"{output_mask_dir}")

VAL_JSON_DIR = r"D:\Study\HumanStudy\Dataset\Validation\02.라벨링데이터"
VAL_MASK_SAVE_DIR = r"D:\Study\HumanStudy\Dataset\Validation_Masks"

generate_validation_masks(VAL_JSON_DIR, VAL_MASK_SAVE_DIR)

52,500files


100%|████████████████████████████████████████████████████████████████████████████| 52500/52500 [09:52<00:00, 88.57it/s]

50,000장
D:\Study\HumanStudy\Dataset\Validation_Masks
